# 02. Financial Ratio Engine & Accounting Integrity Validation

This notebook validates the calculation of financial ratios, inspects double-entry
accounting integrity, and reviews balance sheet reconciliation discrepancies.

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "data").exists():
    project_root = project_root.parent

interim_dir = project_root / "data" / "interim"

## 1. Double-Entry Balance Sheet Identity Audit

The identity Total Assets = Total Liabilities + Shareholders Equity was evaluated
with a 2% tolerance threshold across all company-fiscal years.

In [ ]:
issues_path = interim_dir / "data_quality_issues.csv"
if issues_path.exists():
    df_issues = pd.read_csv(issues_path)
    print(f"Total flagged quality issues: {len(df_issues)}")
    print("Issue types distribution:")
    print(df_issues["issue_type"].value_counts())

## 2. Reconciling Non-Controlling (Minority) Interests

In consolidated reporting, Yahoo Finance maps parent equity into Stockholders Equity,
excluding non-controlling / minority interests. This explains the 273 balance sheet
identity discrepancies observed across the universe.

In [ ]:
if issues_path.exists() and "pct_diff" in df_issues.columns:
    bs_issues = df_issues[df_issues["issue_type"] == "balance_sheet_mismatch"]
    print("Percentage discrepancy distribution for balance sheet identity:")
    print(bs_issues["pct_diff"].describe())

## 3. Ratio Computation Conventions & Safeguards

- Average-balance rule: ROCE and ROE use the average of current and prior fiscal year
  balances when available.
- Negative Capex convention: Free Cash Flow evaluates as FCF = CFO - abs(Capex).
- Denominator guards: Ratios evaluate to NULL on negative equity or zero/negative
  EBIT/PAT to prevent misleading sign reversals.